# Assaying Anomalies Quickstart

This tutorial demonstrates how to use the Python translation of the *Assaying Anomalies* library to compute anomaly signals, sort portfolios, run Fama–MacBeth regressions and interpret the results.

We will work with synthetic data so that anyone can run this notebook without access to proprietary databases. In a real research setting you would load CRSP and Compustat data and follow the same steps.

## Setup

First, import the necessary packages. The `aa` package contains the translated functions for computing signals, performing sorts and running regressions. We'll also import `pandas`, `numpy` and plotting libraries.

In [ ]:
import pandas as pd
import numpy as np
from aa.signals import (
    compute_size_signal,
    compute_book_to_market_signal,
    compute_momentum_signal,
    compute_investment_signal,
    compute_profitability_signal,
)
from aa.asset_pricing.univariate import SortConfig, univariate_sort
from aa.asset_pricing.fama_macbeth import fama_macbeth_full

# Display tables more compactly
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')

## Generate synthetic data

We create a panel of monthly returns for a handful of synthetic stocks over several years. Each firm is assigned random market equity (`me`), book equity (`be`), total assets (`at`) and operating profitability (`op`). Returns are simulated as standard normal draws.

In [ ]:
# Parameters for the synthetic panel
n_firms = 100
n_months = 120
dates = pd.date_range('2000-01-31', periods=n_months, freq='M')
firm_ids = np.arange(1, n_firms + 1)

# Cartesian product of dates and firms
panel = pd.MultiIndex.from_product([dates, firm_ids], names=['date', 'permno']).to_frame(index=False)

# Simulate returns and fundamentals
rng = np.random.default_rng(42)
panel['ret'] = rng.normal(loc=0.01, scale=0.05, size=len(panel))
panel['me'] = rng.lognormal(mean=7, sigma=1.0, size=len(panel))
panel['be'] = rng.lognormal(mean=6.5, sigma=1.0, size=len(panel))
panel['at'] = rng.lognormal(mean=8.0, sigma=1.0, size=len(panel))
panel['op'] = rng.normal(loc=0.1, scale=0.05, size=len(panel)) * panel['be']

# Show the first few rows
panel.head()

## Compute anomaly signals

We compute the five standard anomaly signals using the functions in `aa.signals`. Each function accepts a DataFrame with the required columns and returns a DataFrame with columns `date`, `permno` and `signal`.

* `compute_size_signal` – lagged log market equity.
* `compute_book_to_market_signal` – log book–to–market ratio.
* `compute_momentum_signal` – 12‑minus‑2 momentum.
* `compute_investment_signal` – negative asset growth.
* `compute_profitability_signal` – operating profitability relative to book equity.

In [ ]:
# Compute each signal
size_signal = compute_size_signal(panel[['date', 'permno', 'me']])
bm_signal = compute_book_to_market_signal(panel[['date', 'permno', 'be']], panel[['date', 'permno', 'me']])
mom_signal = compute_momentum_signal(panel[['date', 'permno', 'ret']])
inv_signal = compute_investment_signal(panel[['date', 'permno', 'at']])
prof_signal = compute_profitability_signal(panel[['date', 'permno', 'op']], panel[['date', 'permno', 'be']])

# Merge signals back into the panel for convenience
signals = (size_signal.rename(columns={'signal': 'size'})
          .merge(bm_signal.rename(columns={'signal': 'bm'}), on=['date', 'permno'])
          .merge(mom_signal.rename(columns={'signal': 'momentum'}), on=['date', 'permno'])
          .merge(inv_signal.rename(columns={'signal': 'investment'}), on=['date', 'permno'])
          .merge(prof_signal.rename(columns={'signal': 'profitability'}), on=['date', 'permno']))

panel_signals = panel.merge(signals, on=['date', 'permno'])

panel_signals.head()

## Univariate portfolio sort

To illustrate sorting, we perform a univariate sort on the size signal. We form five portfolios (quintiles) without NYSE breakpoint adjustment and compute equal‑weighted returns and the high‑minus‑low spread.

In [ ]:
# Perform a univariate sort on the size signal
res = univariate_sort(
    returns=panel_signals[['date', 'permno', 'ret']],
    signal=size_signal,
    size=panel_signals[['date', 'permno', 'me']],
    exch=None,
    config=SortConfig(n_bins=5, nyse_breaks=False, min_obs=20),
)

time_series = res['time_series']
summary = res['summary']
hl = res['hl']

# Display the summary table and the first few high-minus-low observations
display(summary.head())
print('High-minus-low first few observations:')
display(hl.head())

## Fama–MacBeth regression

Finally, we estimate a Fama–MacBeth regression with the size and book–to–market signals as regressors. The function `fama_macbeth_full` returns the average beta estimates, their Newey–West standard errors and the full time series of coefficients.

In [ ]:
# Prepare the panel for regression
panel_reg = panel_signals[['date', 'permno', 'ret', 'size', 'bm']].dropna()

reg_res = fama_macbeth_full(
    panel=panel_reg,
    y='ret',
    xcols=['size', 'bm'],
    time_col='date',
    nw_lags=3,
)

print('Average betas:')
display(reg_res['average_beta'])
print('Newey–West standard errors:')
display(reg_res['se_beta'])

## Conclusion

This notebook has illustrated a complete workflow using the Python translation of *Assaying Anomalies*. On synthetic data we computed anomaly signals, formed portfolios, derived high‑minus‑low spreads and estimated Fama–MacBeth regressions. For a real research project, use CRSP and Compustat data in place of the random numbers and explore additional signals or robustness diagnostics. The surrounding documentation and examples provide further guidance on using the package.